# KV Cache & 추론 최적화 - 실습 코드 1: vLLM을 사용한 최적화 추론

- Tutorial ID: `expand-kv-cache`
- Tutorial: KV Cache & 추론 최적화
- Section ID: `expand-kv-cache-code-1`
- Section: 실습 코드 1: vLLM을 사용한 최적화 추론


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: vLLM을 사용한 최적화 추론
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) autoregressive decoding에서 이전 K/V를 재사용해 계산을 줄이는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from vllm import LLM, SamplingParams

# vLLM 초기화 (PagedAttention 자동 활성화)
llm = LLM(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    gpu_memory_utilization=0.90,     # GPU 메모리 90% 사용
    max_num_seqs=256,                # 최대 동시 시퀀스
    enable_prefix_caching=True,      # 공유 프롬프트 캐싱
    kv_cache_dtype="fp8",            # KV Cache 양자화
)

# 배치 추론
sampling_params = SamplingParams(
        temperature=0.7,
    max_tokens=512,
)

prompts = [
    "Explain quantum computing:",
    "What is machine learning?",
    "Describe neural networks:",
]

outputs = llm.generate(prompts, sampling_params)
for output in outputs:
    print(f"Prompt: {output.prompt[:50]}...")
    print(f"Generated: {output.outputs[0].text[:100]}...")
    print(f"Tokens: {len(output.outputs[0].token_ids)}")
    print()